# Azimuthal variance (FEM) analysis of nanobeam diffraction data

## Usage

This notebook takes a 4D-STEM nanobeam electron diffraction (NBED) dataset recorded on the
Merlin detector at ePSIC, calibrates it (diffraction pixel size and elliptical distortion from
an Au cross-grating calibration, descan origin from the data itself), transforms every
diffraction pattern to polar coordinates and computes, at each probe position, the
**azimuthal mean** and **azimuthal variance** of the intensity as a function of scattering
vector *q* (fluctuation electron microscopy statistics).

It is designed to be run non-interactively with papermill by the workflows server; every input
is set in the cell tagged `parameters` below and can be overridden with `-p name value`.

### Parameters

**Data location**

 visit: str

The path to the visit folder (e.g. `/dls/e02/data/2024/mg12345-1`)

 sample: str

The name of the sample (the sub-folder in `processing/Merlin`)

 timestamp: str

The timestamp of the dataset to be processed (`YYYYMMDD_hhmmss`; a space or no separator also works)

 raw_data_path: str

Optional full path to the `_data.hdf5` file. If given, it is used instead of visit/sample/timestamp

 calib_path: str

Folder containing the Au cross-grating calibration `.json` files (searched recursively and matched on nominal camera length)

 maskpath: str

Path to the detector mask (`data/mask` in an h5 file) to apply to the raw data; leave blank for no mask

 outpath: str

Output folder. If blank: `<visit>/processing/workflows/azimuthal_variance/<sample>/<timestamp>`

**Data preparation**

 fill_cross: bool — remove the Merlin cross with `epsic_tools` (needs `/dls_sw/e02/software/epsic_tools`)

 crop_q: int — half-width in pixels of a diffraction-space crop centred on the bright field disc; 0 = no crop

 bin_factor: int — real-space binning factor; 1 = no binning

 v_min, v_max: float — thresholds used to estimate the bright field disc size and position

 hot_pix_thresh: float — threshold for py4DSTEM hot pixel filtering

**Origin (descan) calibration**

 origin_method: str — `'bragg'` (Bragg disk detection at every probe position, then fit; slow) or `'com'` (centre of mass of the bright field disc, then fit; fast)

 origin_fit_function: str — surface fitted to the measured origins (`'plane'`, `'parabola'`, `'bezier_two'`)

 syn_probe_rad, syn_probe_width — radius and edge width (pixels) of the synthetic probe template (`bragg` only)

 probe_kernel_pre, probe_kernel_post — inner/outer sigmoid radii of the correlation kernel, in units of `syn_probe_rad` (`bragg` only)

 peak_finding_minAbsoluteIntensity, peak_finding_minPeakSpacing — Bragg disk detection settings (`bragg` only)

 cross_mask_width: int — half-width (pixels) of the detector cross removed from the Bragg peaks before measuring the origin; 0 = off (`bragg` only)

 adf_inner, adf_outer: int — radii (pixels) of the virtual ADF detector used for the overview image

**Polar transform / variance**

 n_annular: int — number of azimuthal bins

 qstep: float — radial bin size in pixels

 qmax: float — maximum radius in pixels; 0 = automatic

 qscale: float — intensities are multiplied by q**qscale in the polar transform; 0 = off

 use_median: bool — use the median instead of the mean for the azimuthal statistics

 save_prepared_data: bool — also save the prepared (cropped/binned) 4D dataset

### Outputs

Written to `outpath`: `<timestamp>_mean.hspy`, `<timestamp>_variance.hspy`,
`<timestamp>_norm_variance.hspy` (Signal1D, navigation = scan, signal = *q* in Å⁻¹),
`<timestamp>_summary.json` (parameters, calibrations and peak positions) and diagnostic PNG figures.

### Dependencies

- py4DSTEM (0.14)
- hyperspy
- h5py
- numpy, scipy
- matplotlib
- epsic_tools (only if `fill_cross` is set)

In [ ]:
# Parameters (overridden by papermill)
visit = ''
sample = ''
timestamp = ''
raw_data_path = ''
calib_path = ''
maskpath = ''
outpath = ''

# data preparation
fill_cross = False
crop_q = 0
bin_factor = 1
v_min = 0.01
v_max = 0.99
hot_pix_thresh = 0.1

# origin (descan) calibration
origin_method = 'bragg'
origin_fit_function = 'bezier_two'
syn_probe_rad = 4
syn_probe_width = 4.0
probe_kernel_pre = 0.5
probe_kernel_post = 5.0
peak_finding_minAbsoluteIntensity = 0.25
peak_finding_minPeakSpacing = 5
cross_mask_width = 6
adf_inner = 50
adf_outer = 175

# polar transform / variance
n_annular = 360
qstep = 1.0
qmax = 0
qscale = 1.0
use_median = False
save_prepared_data = False

### Imports and logging

In [ ]:
%matplotlib inline
import os
import sys
import io
import glob
import json
import logging

import h5py
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import hyperspy.api as hs
import py4DSTEM
from py4DSTEM.process.calibration.origin import get_origin, fit_origin
from scipy.signal import find_peaks


def console_handler(stream='stdout'):
    # Log to the original console of the kernel, so messages also reach the workflow pod log.
    try:
        fh = getattr(sys, stream)._original_stdstream_copy
        return logging.StreamHandler(io.TextIOWrapper(io.FileIO(fh, 'w')))
    except (AttributeError, TypeError, OSError):
        return logging.StreamHandler(getattr(sys, '__' + stream + '__'))


logger = logging.getLogger('azimuthal_variance')
logger.setLevel(logging.DEBUG)
if not logger.handlers:
    logger.addHandler(console_handler())

logger.debug(f'py4DSTEM {py4DSTEM.__version__} from {py4DSTEM.__file__}')
logger.debug(f'hyperspy {hs.__version__}')

### Tidy up the parameters

papermill infers types from the command line (e.g. `0` is an int, `20231204_150306` becomes the int `20231204150306`), so coerce everything explicitly.

In [ ]:
def to_bool(x):
    if isinstance(x, str):
        return x.strip().lower() in ('1', 'true', 'yes', 'y', 'on')
    return bool(x)


def to_str(x):
    return '' if x is None else str(x).strip()


visit, sample, raw_data_path = to_str(visit), to_str(sample), to_str(raw_data_path)
calib_path, maskpath, outpath = to_str(calib_path), to_str(maskpath), to_str(outpath)

fill_cross = to_bool(fill_cross)
crop_q = int(float(crop_q or 0))
bin_factor = max(1, int(float(bin_factor or 1)))
v_min, v_max = float(v_min), float(v_max)
hot_pix_thresh = float(hot_pix_thresh)

origin_method = to_str(origin_method).lower()
origin_fit_function = to_str(origin_fit_function)
syn_probe_rad = int(float(syn_probe_rad))
syn_probe_width = float(syn_probe_width)
probe_kernel_pre, probe_kernel_post = float(probe_kernel_pre), float(probe_kernel_post)
peak_finding_minAbsoluteIntensity = float(peak_finding_minAbsoluteIntensity)
peak_finding_minPeakSpacing = int(float(peak_finding_minPeakSpacing))
cross_mask_width = int(float(cross_mask_width))
adf_inner, adf_outer = int(float(adf_inner)), int(float(adf_outer))

n_annular = int(float(n_annular))
qstep = float(qstep)
qmax = float(qmax or 0) or None
qscale = float(qscale or 0) or None
use_median = to_bool(use_median)
save_prepared_data = to_bool(save_prepared_data)

if origin_method not in ('bragg', 'com'):
    raise ValueError(f"origin_method must be 'bragg' or 'com', not {origin_method!r}")

# timestamp: accept 'YYYYMMDD_hhmmss', 'YYYYMMDD hhmmss' or 'YYYYMMDDhhmmss'
time_stamp = to_str(timestamp).replace(' ', '_')
if '_' not in time_stamp and len(time_stamp) == 14 and time_stamp.isdigit():
    time_stamp = time_stamp[:8] + '_' + time_stamp[8:]

if raw_data_path == '':
    if not (visit and sample and time_stamp):
        raise ValueError('Give either raw_data_path, or visit, sample and timestamp')
    raw_data_path = os.path.join(visit, 'processing', 'Merlin', sample, time_stamp, f'{time_stamp}_data.hdf5')
elif time_stamp == '':
    time_stamp = os.path.basename(raw_data_path)[:15]

if not raw_data_path.endswith('_data.hdf5'):
    raise ValueError(f'Expected a *_data.hdf5 file, got {raw_data_path}')
meta_path = raw_data_path[:-len('_data.hdf5')] + '.hdf'

if outpath == '':
    if not visit:
        raise ValueError('Give outpath, or visit so a default output folder can be made')
    outpath = os.path.join(visit, 'processing', 'workflows', 'azimuthal_variance', sample or 'sample', time_stamp)
os.makedirs(outpath, exist_ok=True)

for name in ('raw_data_path', 'meta_path', 'outpath', 'time_stamp', 'origin_method'):
    logger.debug(f'{name}: {globals()[name]}')


def savefig(name, fig=None):
    (fig or plt.gcf()).savefig(os.path.join(outpath, name), dpi=150, bbox_inches='tight')

### Load the metadata and the matching Au calibration

In [ ]:
with h5py.File(meta_path, 'r') as f:
    md_group = f['metadata']
    print('Metadata keys:', list(md_group.keys()))
    acc_voltage = float(md_group['ht_value(V)'][()])
    nominal_CL = float(md_group['nominal_camera_length(m)'][()])
    probe_step_size_Ang = float(md_group['step_size(m)'][()]) * 1e10

print(f'HT: {acc_voltage / 1e3:g} kV, nominal CL: {nominal_CL} m, step size: {probe_step_size_Ang:.3f} A')

In [ ]:
json_calibration_files = sorted(glob.glob(os.path.join(calib_path, '**', '*.json'), recursive=True))
if not json_calibration_files:
    raise FileNotFoundError(f'No calibration .json files found under calib_path={calib_path!r}')

cal_dict = None
for this_cal_file in json_calibration_files:
    with open(this_cal_file, 'r') as fp:
        this_dict = json.load(fp)
    if 'nominal_camera_length(m)' in this_dict and np.isclose(this_dict['nominal_camera_length(m)'], nominal_CL):
        cal_dict, cal_file = this_dict, this_cal_file

if cal_dict is None:
    raise ValueError(f'None of {json_calibration_files} is for nominal camera length {nominal_CL} m')

pixel_size_inv_Ang = float(cal_dict['reciprocal_space_pix(1/A)'])
p_ellipse = [float(p) for p in cal_dict['p_ellipse']]
logger.debug(f'calibration file: {cal_file}')
print('pixel_size_inv_Ang :', pixel_size_inv_Ang)
print('p_ellipse :', p_ellipse)

### Prepare the data

Load, optionally mask / remove the cross / crop in q / bin in real space.

In [ ]:
d = hs.load(raw_data_path, reader='HSPY')
data = d.data
del d
logger.debug(f'raw dataset shape after loading: {data.shape}')

# detector mask for hot / dead pixels (same convention as the ptycho calibration notebook)
if maskpath:
    with h5py.File(maskpath, 'r') as f:
        det_mask = np.invert(f['data/mask'][()].astype(bool))
    if det_mask.shape == data.shape[-2:]:
        data *= det_mask  # in place: the raw data can be tens of GB
    else:
        logger.warning(f'mask shape {det_mask.shape} does not match detector {data.shape[-2:]}; mask not applied')

if fill_cross:
    sys.path.append('/dls_sw/e02/software/epsic_tools')
    import epsic_tools.api as epsic
    nav_shape = data.shape[:-2]
    frames = data.reshape((-1,) + data.shape[-2:])
    data = np.stack([epsic.warp_3d.remove_cross(fr) for fr in frames])
    data = data.reshape(nav_shape + data.shape[-2:])

# a flat stack of frames is a square scan with a fly-back column, which is dropped
if data.ndim == 3:
    s1 = int(np.floor(data.shape[0] ** 0.5))
    s2 = data.shape[0] // s1
    data = data[:s1 * s2].reshape((s1, s2) + data.shape[-2:])[:, :-1]

det_shape_raw = data.shape[-2:]
dp_mean_raw = data.mean(axis=(0, 1))
probe_semiangle, qx0, qy0 = py4DSTEM.process.calibration.get_probe_size(dp_mean_raw, v_min, v_max)
logger.debug(f'bright field disc radius and centre in the raw data: {probe_semiangle, qx0, qy0}')

fig, ax = py4DSTEM.show(dp_mean_raw, cmap='inferno', vmax=0.9, returnfig=True, title='mean raw DP')
savefig('mean_raw_diff.png', fig)

# crop in diffraction space around the bright field disc; remember the offset of the crop
crop_offset = (0, 0)
if crop_q > 0:
    x0 = max(0, int(qx0 - crop_q))
    y0 = max(0, int(qy0 - crop_q))
    data = data[..., x0:int(qx0 + crop_q), y0:int(qy0 + crop_q)]
    crop_offset = (x0, y0)
    logger.debug(f'cropped data to {data.shape}')

# bin in real space
if bin_factor > 1:
    nx, ny = (s // bin_factor * bin_factor for s in data.shape[:2])
    data = data[:nx, :ny].reshape(
        nx // bin_factor, bin_factor, ny // bin_factor, bin_factor, *data.shape[-2:]
    ).sum(axis=(1, 3))
    probe_step_size_Ang *= bin_factor
    logger.debug(f'binned data to {data.shape}')

dataset = py4DSTEM.DataCube(data)
del data
logger.debug(f'prepared dataset shape: {dataset.shape}')

if save_prepared_data:
    py4DSTEM.save(os.path.join(outpath, f'{time_stamp}_prepared_data.h5'), dataset, mode='o')

### Calibrate pixel sizes and remove hot pixels

In [ ]:
dataset.calibration.set_Q_pixel_size(pixel_size_inv_Ang)
dataset.calibration.set_Q_pixel_units('A^-1')
dataset.calibration.set_R_pixel_size(probe_step_size_Ang)
dataset.calibration.set_R_pixel_units('A')

dataset, hot_mask = dataset.filter_hot_pixels(thresh=hot_pix_thresh, return_mask=True)
fig, ax = py4DSTEM.show(hot_mask, returnfig=True, title=f'hot pixels ({int(hot_mask.sum())})')
savefig('hot_pix_mask.png', fig)

dataset.get_dp_max()
dataset.get_dp_mean()
fig, ax = py4DSTEM.show(
    [dataset.tree('dp_mean'), dataset.tree('dp_max')],
    cmap='inferno', scaling='power', power=0.5, returnfig=True,
)
savefig('dp_mean_max.png', fig)

probe_semiangle, probe_qx0, probe_qy0 = py4DSTEM.process.calibration.get_probe_size(
    dataset.tree('dp_mean').data, v_min, v_max
)
print(f'Bright field disc radius: {probe_semiangle:.2f} px, centre: ({probe_qx0:.2f}, {probe_qy0:.2f})')

### Virtual ADF overview image

In [ ]:
radii = (adf_inner, adf_outer)
fig, ax = py4DSTEM.show(
    dataset.tree('dp_max'), scaling='log',
    annulus={'center': (probe_qx0, probe_qy0), 'radii': radii, 'alpha': 0.3, 'fill': True},
    returnfig=True,
)
savefig('adf_detector.png', fig)

dataset.get_virtual_image(
    mode='annulus', geometry=((probe_qx0, probe_qy0), radii), name='dark_field',
)
fig, ax = py4DSTEM.show(dataset.tree('dark_field'), returnfig=True, title='virtual ADF')
savefig('virtual_adf.png', fig)

### Measure and fit the origin of every diffraction pattern (descan)

#### `bragg`: detect Bragg disks with a synthetic probe, mask the cross, measure the origin

In [ ]:
if origin_method == 'bragg':
    # test the detection on a few bright probe positions first
    rng = np.random.default_rng(0)
    df = dataset.tree('dark_field').data
    pos = np.argwhere(df > df.mean())
    rxs, rys = pos[rng.choice(len(pos), size=min(4, len(pos)), replace=False)].T
    rxs, rys = tuple(int(r) for r in rxs), tuple(int(r) for r in rys)

    syn_probe = py4DSTEM.braggvectors.probe.Probe.generate_synthetic_probe(
        syn_probe_rad, syn_probe_width, dataset.data.shape[-2:]
    )
    syn_probe_kernel = syn_probe.get_kernel(
        mode='sigmoid',
        radii=(syn_probe_rad * probe_kernel_pre, syn_probe_rad * probe_kernel_post),
        bilinear=True,
    )
    fig, ax = py4DSTEM.visualize.show_kernel(syn_probe.kernel, R=20, L=20, W=1, figsize=(8, 4), returnfig=True)
    savefig('probe_kernel.png', fig)

    detect_params = {
        'corrPower': 1.0,
        'sigma': 0,
        'edgeBoundary': 2,
        'minRelativeIntensity': 0.0,
        'minAbsoluteIntensity': peak_finding_minAbsoluteIntensity,
        'minPeakSpacing': peak_finding_minPeakSpacing,
        'subpixel': 'poly',
        'upsample_factor': 2,
        'maxNumPeaks': 1000,
    }

    disks_selected = dataset.find_Bragg_disks(data=(rxs, rys), template=syn_probe_kernel, **detect_params)
    n_test = len(rxs)
    fig, ax = py4DSTEM.visualize.show_image_grid(
        get_ar=lambda i: dataset.data[rxs[i], rys[i]],
        H=1, W=n_test, axsize=(3, 3), vmax=0.98, returnfig=True,
        get_x=lambda i: disks_selected[i].data['qx'],
        get_y=lambda i: disks_selected[i].data['qy'],
        open_circles=True, scale=50,
    )
    savefig('peak_finding_test.png', fig)
else:
    print('origin_method is com: skipping Bragg disk detection')

In [ ]:
if origin_method == 'bragg':
    bragg_peaks = dataset.find_Bragg_disks(template=syn_probe_kernel, **detect_params)
    bragg_peaks.setcal()

    # remove peaks on the detector cross (centre of the raw detector, shifted by any crop)
    mask_cross = np.zeros(dataset.data.shape[-2:], dtype=bool)
    if cross_mask_width > 0:
        cx = int(det_shape_raw[0] / 2 - crop_offset[0])
        cy = int(det_shape_raw[1] / 2 - crop_offset[1])
        w = cross_mask_width
        if 0 <= cx < mask_cross.shape[0]:
            mask_cross[max(0, cx - w):cx + w, :] = True
        if 0 <= cy < mask_cross.shape[1]:
            mask_cross[:, max(0, cy - w):cy + w] = True
        # never mask the unscattered beam itself, or the origin would lock onto a Bragg disk
        qxx, qyy = np.indices(mask_cross.shape)
        keep = np.hypot(qxx - probe_qx0, qyy - probe_qy0) < 2 * probe_semiangle + w
        if (mask_cross & keep).any():
            logger.warning('the bright field disc overlaps the detector cross; not masking the cross there')
        mask_cross &= ~keep
    bragg_peaks_masked = bragg_peaks.mask_in_Q(mask_cross)

    fig, ax = py4DSTEM.show(
        [bragg_peaks.histogram(mode='raw'), bragg_peaks_masked.histogram(mode='raw')],
        returnfig=True, title=['BVM', 'BVM, cross masked'],
    )
    savefig('bragg_peaks_masked.png', fig)

    qx0_meas, qy0_meas, mask_meas = bragg_peaks_masked.measure_origin(
        center_guess=(probe_qx0, probe_qy0),
        score_method='intensity weighted distance',
        findcenter='max',
    )

#### `com`: centre of mass of the bright field disc

In [ ]:
if origin_method == 'com':
    qx0_meas, qy0_meas, mask_meas = get_origin(dataset, r=probe_semiangle)

#### Fit a smooth surface to the measured origins

In [ ]:
fig, ax = py4DSTEM.show([qx0_meas, qy0_meas], cmap='RdBu', returnfig=True, title=['qx0 measured', 'qy0 measured'])
savefig('origin_measured.png', fig)

# only fit positions where an origin was found (True = use); failed positions come back as 0
# (py4DSTEM's measure_origin returns a scalar False if any position had no peaks, so rebuild it)
valid = np.ones(np.shape(qx0_meas), dtype=bool)
if mask_meas is not None and np.ndim(mask_meas) == 2:
    valid &= np.asarray(mask_meas, dtype=bool)
valid &= (np.asarray(qx0_meas) != 0) & (np.asarray(qy0_meas) != 0)
logger.debug(f'origin found at {int(valid.sum())} / {valid.size} probe positions')

qx0_fit, qy0_fit, qx0_residuals, qy0_residuals = fit_origin(
    (np.asarray(qx0_meas), np.asarray(qy0_meas)), mask=valid,
    fitfunction=origin_fit_function, robust=True,
)
qx0_residuals = np.where(valid, qx0_residuals, 0)
qy0_residuals = np.where(valid, qy0_residuals, 0)

fig, ax = py4DSTEM.show(
    [qx0_meas, qx0_fit, qx0_residuals, qy0_meas, qy0_fit, qy0_residuals],
    cmap='RdBu', returnfig=True, figsize=(12, 3),
    title=['qx0 meas', 'qx0 fit', 'qx0 residuals', 'qy0 meas', 'qy0 fit', 'qy0 residuals'],
)
savefig('BF_disc_align.png', fig)
print(f'origin residuals (rms, px): qx0 {np.sqrt(np.mean(qx0_residuals ** 2)):.3f}, '
      f'qy0 {np.sqrt(np.mean(qy0_residuals ** 2)):.3f}')

### Apply the origin and elliptical calibrations to the dataset

In [ ]:
dataset.calibration.set_qx0(qx0_fit)
dataset.calibration.set_qy0(qy0_fit)
dataset.calibration.set_p_ellipse(p_ellipse)  # only a, b, theta are used; the origin is left alone
dataset.calibrate()
logger.debug(f'dataset calibration: {dataset.calibration}')

im_align = dataset.get_virtual_diffraction(method='mean', shift_center=True)
fig, ax = py4DSTEM.show(
    [dataset.tree('dp_mean'), im_align], cmap='inferno', scaling='power', power=0.5,
    returnfig=True, title=['mean DP', 'mean DP, descan corrected'],
)
savefig('dp_mean_aligned.png', fig)

### Polar transform

In [ ]:
polardata = py4DSTEM.PolarDatacube(
    dataset,
    qmin=0,
    qmax=qmax,
    qstep=qstep,
    n_annular=n_annular,
    ellipse=True,
    qscale=qscale,
)

rx, ry = dataset.shape[0] // 2, dataset.shape[1] // 2
fig, ax = py4DSTEM.show(
    [dataset.data[rx, ry], polardata.data[rx, ry]],
    axsize=(4, 4), ticks=False, cmap='turbo', returnfig=True,
    title=[f'DP ({rx}, {ry})', 'polar transform'],
)
savefig('polar_transform_example.png', fig)

### Azimuthal mean and variance at every probe position

In [ ]:
local_radial_mean, local_radial_var, fig, ax = polardata.calculate_FEM_local(
    use_median=use_median,
    plot_normalized_variance=True,
    return_values=True,
    returnfig=True,
)
savefig('mean_normalised_variance.png', fig)

with np.errstate(divide='ignore', invalid='ignore'):
    local_norm_var = np.where(local_radial_mean > 0, local_radial_var / local_radial_mean ** 2, 0.0)

### Save the results as HyperSpy signals

In [ ]:
q_axis = np.asarray(polardata.qq, dtype=float)  # in A^-1
q_offset = float(q_axis[0])
q_scale = float(q_axis[1] - q_axis[0]) if q_axis.size > 1 else pixel_size_inv_Ang


def to_signal(arr, title):
    s = hs.signals.Signal1D(np.asarray(arr, dtype=np.float32))
    s.metadata.General.title = title
    s.metadata.General.original_filename = os.path.basename(raw_data_path)
    for ax_, name in zip(s.axes_manager.navigation_axes, ('x', 'y')):
        ax_.name, ax_.scale, ax_.units = name, probe_step_size_Ang, 'A'
    sig = s.axes_manager.signal_axes[0]
    sig.name, sig.scale, sig.offset, sig.units = 'Scattering vector', q_scale, q_offset, '1/A'
    return s


results = {
    'mean': to_signal(local_radial_mean, 'azimuthal mean'),
    'variance': to_signal(local_radial_var, 'azimuthal variance'),
    'norm_variance': to_signal(local_norm_var, 'normalised azimuthal variance'),
}
for key, s in results.items():
    s.save(os.path.join(outpath, f'{time_stamp}_{key}.hspy'), overwrite=True)
    logger.debug(f'saved {key}: {s}')

### Summed profiles and peak positions

In [ ]:
sum_mean = local_radial_mean.sum(axis=(0, 1))
sum_var = local_radial_var.sum(axis=(0, 1))
mean_norm_var = local_norm_var.mean(axis=(0, 1))

peak_q = {}
fig, axs = plt.subplots(3, 1, figsize=(8, 9), sharex=True)
for ax, (key, profile) in zip(axs, [('mean', sum_mean), ('variance', sum_var), ('norm_variance', mean_norm_var)]):
    idx, _ = find_peaks(profile, prominence=0.02 * (profile.max() - profile.min()))
    peak_q[key] = q_axis[idx].tolist()
    ax.plot(q_axis, profile)
    for q in q_axis[idx]:
        ax.axvline(q, ls=':', lw=0.8, c='k')
    ax.set_ylabel(key)
axs[-1].set_xlabel('q (1/A)')
fig.tight_layout()
savefig('profiles_peaks.png', fig)

for key, qs in peak_q.items():
    print(f'{key} peaks (1/A):', np.round(qs, 4))

### Normalised variance map at the strongest variance peak

In [ ]:
i_peak = int(np.argmax(np.where(q_axis > q_axis[0] + 2 * probe_semiangle * pixel_size_inv_Ang, mean_norm_var, -np.inf)))
fig, axs = plt.subplots(1, 2, figsize=(10, 4))
axs[0].imshow(dataset.tree('dark_field').data, cmap='gray')
axs[0].set_title('virtual ADF')
im = axs[1].imshow(local_norm_var[:, :, i_peak], cmap='viridis')
axs[1].set_title(f'normalised variance at q = {q_axis[i_peak]:.3f} 1/A')
fig.colorbar(im, ax=axs[1])
savefig('norm_variance_map.png', fig)

### Write a summary for provenance

In [ ]:
summary = {
    'raw_data_path': raw_data_path,
    'meta_path': meta_path,
    'calibration_file': cal_file,
    'time_stamp': time_stamp,
    'py4DSTEM_version': py4DSTEM.__version__,
    'hyperspy_version': hs.__version__,
    'acc_voltage(V)': acc_voltage,
    'nominal_camera_length(m)': nominal_CL,
    'probe_step_size(A)': probe_step_size_Ang,
    'reciprocal_space_pix(1/A)': pixel_size_inv_Ang,
    'p_ellipse': p_ellipse,
    'prepared_shape': list(dataset.shape),
    'crop_offset': list(crop_offset),
    'bright_field_disc': {'radius': float(probe_semiangle), 'qx0': float(probe_qx0), 'qy0': float(probe_qy0)},
    'parameters': {
        'fill_cross': fill_cross, 'crop_q': crop_q, 'bin_factor': bin_factor,
        'v_min': v_min, 'v_max': v_max, 'hot_pix_thresh': hot_pix_thresh,
        'origin_method': origin_method, 'origin_fit_function': origin_fit_function,
        'syn_probe_rad': syn_probe_rad, 'syn_probe_width': syn_probe_width,
        'probe_kernel_pre': probe_kernel_pre, 'probe_kernel_post': probe_kernel_post,
        'peak_finding_minAbsoluteIntensity': peak_finding_minAbsoluteIntensity,
        'peak_finding_minPeakSpacing': peak_finding_minPeakSpacing,
        'cross_mask_width': cross_mask_width, 'n_annular': n_annular, 'qstep': qstep,
        'qmax': qmax, 'qscale': qscale, 'use_median': use_median,
    },
    'peak_q(1/A)': peak_q,
}
with open(os.path.join(outpath, f'{time_stamp}_summary.json'), 'w') as fp:
    json.dump(summary, fp, indent=2)
logger.debug(f'Finished. Results in {outpath}')
print('Results written to', outpath)